<div dir="rtl" align="right">

# رگرسیون لجستیک

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
با وجود اسمش، رگرسیون لجستیک یک مدل **طبقه‌بندی** است. ابتدا مثل رگرسیون خطی یک امتیاز می‌سازد، بعد آن را با تابع **سیگموید** به احتمالی بین ۰ و ۱ تبدیل می‌کند:

$$z=b+w_1x_1+\dots+w_dx_d,\qquad p=\sigma(z)=\frac{1}{1+e^{-z}}$$

اگر $p\ge0.5$ (یعنی $z\ge0$)، کلاس ۱. پس مرز تصمیم $z=0$ است: یک **خط** (یا صفحه).

[[lab:sigmoid]]

| $z$ | $\sigma(z)$ |
|---|---|
| −4 | 0.018 |
| −1 | 0.269 |
| 0 | 0.5 |
| 1 | 0.731 |
| 4 | 0.982 |

## تفسیر ضرایب: نسبت شانس
$z$ لگاریتم **شانس** است: $z=\ln\frac{p}{1-p}$. اگر $w_j=0.7$ باشد، با یک واحد افزایش $x_j$، شانس $e^{0.7}\approx2$ برابر می‌شود.

## تابع هزینه: آنتروپی متقاطع
MSE برای احتمال مناسب نیست. از **لاگ‌لاس** استفاده می‌کنیم که پیش‌بینی مطمئن ولی غلط را به شدت جریمه می‌کند:

$$J=-\frac1n\sum_i\big[y_i\ln p_i+(1-y_i)\ln(1-p_i)\big]$$

> [چپ: لاگ‌لاس؛ وقتی کلاس واقعی ۱ است و مدل می‌گوید p≈0، خطا بی‌نهایت می‌شود. راست: با ویژگی‌های چندجمله‌ای، مرز لجستیک منحنی می‌شود.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/logit_loss.png)

*چپ: لاگ‌لاس؛ وقتی کلاس واقعی ۱ است و مدل می‌گوید p≈0، خطا بی‌نهایت می‌شود. راست: با ویژگی‌های چندجمله‌ای، مرز لجستیک منحنی می‌شود.*

گرادیان این تابع ساده است: $\nabla_w J=\frac1n X^\top(p-y)$؛ با گرادیان کاهشی کمینه می‌شود.

## مثال عددی
$w=2$، $b=-3$. برای $x=2$: $z=1$ و $p=0.731$ ← کلاس ۱. اگر برچسب واقعی ۱ باشد، خطا $-\ln0.731=0.313$ است؛ اگر ۰ باشد، $-\ln0.269=1.313$.



</div>

In [ ]:
import numpy as np

# Logistic regression from scratch with gradient descent
rng = np.random.default_rng(0)
hours = rng.uniform(0, 10, 100)                       # study hours
passed = (hours + rng.normal(0, 1.5, 100) > 5).astype(int)

x = (hours - hours.mean()) / hours.std()              # scaling helps gradient descent
w, b, lr = 0.0, 0.0, 0.5
for step in range(500):
    p = 1 / (1 + np.exp(-(w * x + b)))
    w -= lr * np.mean((p - passed) * x)
    b -= lr * np.mean(p - passed)
loss = -np.mean(passed * np.log(p) + (1 - passed) * np.log(1 - p))
print(f"w = {w:.3f}, b = {b:.3f}, log loss = {loss:.3f}")
h = 6
print("P(pass | 6 hours) =", round(1 / (1 + np.exp(-(w * (h - hours.mean()) / hours.std() + b))), 3))
print("Odds multiply by", round(np.exp(w / hours.std()), 2), "for each extra hour")

<div dir="rtl" align="right">



## آزمایشگاه مرز تصمیم
با درجه ۱ مرز خط راست است. روی داده «دو هلال» درجه را ۳ کن. `C` عکس قدرت منظم‌سازی است: C کوچک یعنی منظم‌سازی قوی.

[[lab:cls_logistic]]

## چندکلاسه: Softmax
برای $K$ کلاس، برای هر کلاس یک امتیاز $z_k$ می‌سازیم و احتمال‌ها:

$$p_k=\frac{e^{z_k}}{\sum_j e^{z_j}}$$

| هایپرپارامتر | معنی |
|---|---|
| `C` | عکس منظم‌سازی (پیش‌فرض ۱)؛ کوچک‌تر = ساده‌تر |
| `penalty` | `l2` (پیش‌فرض)، `l1` (انتخاب ویژگی)، `elasticnet` |
| `class_weight` | `balanced` برای داده نامتوازن |
| `max_iter` | اگر هشدار همگرایی گرفتی زیادش کن |



</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

iris = load_iris()
model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0)).fit(iris.data, iris.target)
print("Accuracy (5-fold):", cross_val_score(model, iris.data, iris.target, cv=5).mean().round(3))
probs = model.predict_proba(iris.data[[0, 75, 140]])
for row, name in zip(probs.round(3), ["sample 0", "sample 75", "sample 140"]):
    print(name, dict(zip(iris.target_names, row)), "sum =", row.sum().round(3))

<div dir="rtl" align="right">



| مزایا | معایب |
|---|---|
| احتمال خوش‌کالیبره | فقط مرز خطی (مگر ویژگی بسازی) |
| سریع و قابل تفسیر | به مقیاس‌بندی نیاز دارد |
| پایه عالی برای مقایسه | با ویژگی‌های همبسته ناپایدار |

</div>

<div dir="rtl" align="right">

### تمرین

احتمال و لاگ‌لاس مثال درس (w=2، b=-3، x=2) را با numpy حساب کن.

</div>

In [ ]:
# Your answer here

<div dir="rtl" align="right">

### تمرین

روی داده سرطان سینه، LogisticRegression با penalty='l1' و solver='liblinear' و C=0.1 بساز. چند ضریب صفر شد؟

</div>

In [ ]:
# Your answer here